# Power Rankings Generator
Pulls team/roster data, calls Anthropic API, writes `power_rankings.json`.

In [1]:
# ============================================================
# 1. Imports, .env loading, and config
# ============================================================

import os
import json
import re
import requests
from datetime import datetime, timezone

# Load .env from repo root — find_dotenv searches upward from CWD
# so it finds ../wilsons-moms-house/.env when CWD is notebooks/
try:
    from dotenv import load_dotenv, find_dotenv
    _dotenv_path = find_dotenv(usecwd=True)
    print(f'Looking for .env — found: {_dotenv_path or "not found"}')
    if _dotenv_path:
        load_dotenv(_dotenv_path)
        print('Loaded .env ✅')
    else:
        print('No .env file found in directory tree')
except ImportError:
    print('python-dotenv not installed — skipping .env load')

LEAGUE_ID = '1312130103358021632'

# Detect data directory: works whether CWD is notebooks/ or repo root
_cwd = os.getcwd()
_candidate_notebooks = os.path.normpath(os.path.join(_cwd, '..', 'public', 'data'))
_candidate_root      = os.path.normpath(os.path.join(_cwd, 'public', 'data'))
DATA_DIR   = _candidate_notebooks if os.path.isdir(_candidate_notebooks) else _candidate_root
OUTPUT_DIR = DATA_DIR

print(f'CWD: {_cwd}')
print(f'Data directory: {DATA_DIR}')
print('Config loaded ✅')

Looking for .env — found: /Users/evankleiner/wilsons-moms-house/.env
Loaded .env ✅
CWD: /Users/evankleiner/wilsons-moms-house/notebooks
Data directory: /Users/evankleiner/wilsons-moms-house/public/data
Config loaded ✅


In [2]:
# ============================================================
# 2. Check ANTHROPIC_API_KEY and initialize client
# ============================================================

import anthropic

api_key = os.environ.get('ANTHROPIC_API_KEY')

if not api_key:
    print()
    print('⚠️  WARNING: ANTHROPIC_API_KEY not found in environment.')
    print()
    print('To fix this locally, create a .env file in the repo root or notebooks/ directory:')
    print()
    print('    ANTHROPIC_API_KEY=your-api-key-here')
    print()
    print('To fix this in GitHub Actions, add ANTHROPIC_API_KEY as a repository secret:')
    print('  1. Go to your repo Settings → Secrets and variables → Actions')
    print('  2. Click "New repository secret"')
    print('  3. Name: ANTHROPIC_API_KEY  |  Value: your API key')
    print()
    raise ValueError('ANTHROPIC_API_KEY not set — cannot generate power rankings.')

client = anthropic.Anthropic(api_key=api_key)
print('Anthropic client initialized ✅')

Anthropic client initialized ✅


In [3]:
# ============================================================
# 3. Pull Sleeper data (users, rosters, team names)
# ============================================================

print('Fetching Sleeper league data...')

users   = requests.get(f'https://api.sleeper.app/v1/league/{LEAGUE_ID}/users').json()
rosters = requests.get(f'https://api.sleeper.app/v1/league/{LEAGUE_ID}/rosters').json()

# Build user map: user_id -> {display_name, team_name}
user_info = {}
for u in users:
    display = u.get('display_name') or u.get('username', 'Unknown')
    team    = (u.get('metadata') or {}).get('team_name') or display
    user_info[u['user_id']] = {'display_name': display, 'team_name': team}

# Build map: owner display_name -> Sleeper team name
owner_team_map = {}
for r in rosters:
    info    = user_info.get(r['owner_id'], {})
    display = info.get('display_name', 'Unknown')
    team    = info.get('team_name', display)
    owner_team_map[display] = team

print(f'Pulled {len(users)} users, {len(rosters)} rosters ✅')
for owner, team in owner_team_map.items():
    print(f'  {owner} \u2192 team name: {team}')

Fetching Sleeper league data...


Pulled 10 users, 10 rosters ✅
  ekleiner1123 → team name: Maye I Skattebust in you
  Herschey6153 → team name: Osama Bin Madden 💍
  jsinykin → team name: Siny
  SvenMoney34 → team name: The Swenvengers
  Akracoon → team name: Skyline Chili Enjoyer
  GiantHawkTua → team name: GiantHawkTua
  nchernandez19 → team name: Mother Loving Quathes
  sethfriedman12 → team name: Rec-Specs
  GreyWaedekin27 → team name: LAMBs to the Slaughter💍
  gavinw20 → team name: 00pium


In [4]:
# ============================================================
# 4. Read existing JSON files from public/data/
# ============================================================

print('Reading existing JSON data...')

def read_json(filename):
    path = os.path.join(DATA_DIR, filename)
    with open(path) as f:
        return json.load(f)

team_overview         = read_json('teamOverview.json')
player_universe       = read_json('playerUniverse.json')
roster_grades         = read_json('rosterGrades.json')
trade_history         = read_json('tradeHistory.json')
positional_proportion = read_json('positionalProportion.json')

print(f'  teamOverview:         {len(team_overview)} teams')
print(f'  playerUniverse:       {len(player_universe)} players')
print(f'  rosterGrades:         {len(roster_grades)} teams')
print(f'  tradeHistory:         {len(trade_history)} trades')
print(f'  positionalProportion: {len(positional_proportion)} teams')
print('JSON files read ✅')

Reading existing JSON data...
  teamOverview:         10 teams
  playerUniverse:       274 players
  rosterGrades:         10 teams
  tradeHistory:         75 trades
  positionalProportion: 10 teams
JSON files read ✅


In [5]:
# ============================================================
# 5. Build per-team data structure for the prompt
# ============================================================

TIER_ORDER = [
    'Cornerstone', 'Foundational', 'Upside Premier', 'Mainstay',
    'Productive Vet', 'Short-term Winner', 'Upside Shot',
    'Short-term Production', 'Serviceable',
    'Jag Developmental', 'Jag Insurance', 'Replaceable',
]

# Index lookups
grades_by_owner     = {t['Owner']: t for t in roster_grades}
proportion_by_owner = {t['Owner']: t for t in positional_proportion}

def get_top_players(owner, n=5):
    players = [p for p in player_universe if p.get('Dynasty Owner') == owner]
    players.sort(key=lambda p: p.get('KTC Value') or 0, reverse=True)
    return [
        {
            'name':     p['Player'],
            'position': p['Position'],
            'ktc':      int(p.get('KTC Value') or 0),
            'tier':     p.get('Tier', 'Unknown'),
        }
        for p in players[:n]
    ]

def get_tier_breakdown(owner):
    players = [p for p in player_universe if p.get('Dynasty Owner') == owner]
    breakdown = {}
    for p in players:
        t = p.get('Tier', 'Unknown')
        breakdown[t] = breakdown.get(t, 0) + 1
    return {t: breakdown[t] for t in TIER_ORDER if t in breakdown}

def get_positional_depth(owner):
    players = [p for p in player_universe if p.get('Dynasty Owner') == owner]
    depth = {'QB': [], 'RB': [], 'WR': [], 'TE': []}
    for p in players:
        pos = p.get('Position')
        if pos in depth:
            depth[pos].append({
                'name': p['Player'],
                'tier': p.get('Tier', 'Unknown'),
                'ktc':  int(p.get('KTC Value') or 0),
            })
    for pos in depth:
        depth[pos].sort(key=lambda x: x['ktc'], reverse=True)
    return {
        pos: [{'name': p['name'], 'tier': p['tier']} for p in depth[pos][:4]]
        for pos in depth
    }

def get_recent_trades(owner, n=4):
    trades = [
        t for t in trade_history
        if t.get('Team A') == owner or t.get('Team B') == owner
    ]
    trades.sort(key=lambda t: t.get('Date', ''), reverse=True)
    result = []
    for t in trades[:n]:
        is_a    = t['Team A'] == owner
        received = t['Team A Received'] if is_a else t['Team B Received']
        gave     = t['Team B Received'] if is_a else t['Team A Received']
        surplus  = int(t['Surplus A'] if is_a else t['Surplus B'] or 0)
        result.append({
            'date':     t.get('Date', ''),
            'opponent': t['Team B'] if is_a else t['Team A'],
            'received': received,
            'gave':     gave,
            'surplus':  surplus,
        })
    return result

teams_data = []
for team in team_overview:
    owner  = team['Owner']
    grades = grades_by_owner.get(owner, {})
    prop   = proportion_by_owner.get(owner, {})

    team_info = {
        'owner':        owner,
        'team_name':    owner_team_map.get(owner, owner),
        'outlook':      team['Outlook'],
        'value_rank':   team['Value Rank'],
        'cf_total':     team['C+F Total'],
        'total_value':  int(team.get('Total Value') or 0),
        'player_value': int(team.get('Player Value') or 0),
        'pick_value':   int(team.get('Pick Value') or 0),
        'value_pct':    team.get('Value Share %', 0),
        'production_pct': team.get('Production Share %', 0),
        'total_1sts':   team.get('Total 1sts', 0),
        'positional_grades': {
            'QB': round(grades.get('QB Grade') or 0),
            'RB': round(grades.get('RB Grade') or 0),
            'WR': round(grades.get('WR Grade') or 0),
            'TE': round(grades.get('TE Grade') or 0),
        },
        'top_players':           get_top_players(owner),
        'tier_breakdown':        get_tier_breakdown(owner),
        'positional_depth':      get_positional_depth(owner),
        'recent_trades':         get_recent_trades(owner),
        'positional_proportion': {
            'QB': prop.get('QB %', 0),
            'RB': prop.get('RB %', 0),
            'WR': prop.get('WR %', 0),
            'TE': prop.get('TE %', 0),
        },
    }
    teams_data.append(team_info)
    print(f'  Built data for {owner} ({team["Outlook"]})')

print(f'\n✅ Built data for {len(teams_data)} teams')

  Built data for jsinykin (Reload)
  Built data for ekleiner1123 (Contender)
  Built data for SvenMoney34 (Contender)
  Built data for sethfriedman12 (Rebuild (future value))
  Built data for GiantHawkTua (Contender)
  Built data for gavinw20 (Reload)
  Built data for nchernandez19 (Rebuild (future value))
  Built data for Akracoon (Rebuild (future value))
  Built data for GreyWaedekin27 (Window Contender)
  Built data for Herschey6153 (Window Contender)

✅ Built data for 10 teams


In [6]:
# ============================================================
# 6. Format prompt and call Anthropic API
# ============================================================

DYNASTYZE_REFERENCE = '''Dynastyze current power rankings (for calibration reference only):
1. Maye I Skattebust in you (ekleiner1123) — 77.9K
2. The Swenvengers (SvenMoney34) — 73.1K
3. GiantHawkTua (GiantHawkTua) — 66.6K
4. 00pium (gavinw20) — 64.3K
5. Siny (jsinykin) — 61.6K
6. LAMBs to the Slaughter (Herschey6153) — 53.7K
7. Osama Bin Madden (nchernandez19) — 52.9K
8. Mother Loving Quathes (GreyWaedekin27) — 52.3K
9. Rec-Specs (Akracoon) — 51.2K
10. Skyline Chili Enjoyer (sethfriedman12) — 28.4K

Use these Dynastyze rankings as a soft reference point — they reflect a reasonable external \
view of team strength. Form your own independent ranking based on the data provided, but \
consider this calibration when teams are close.'''

SYSTEM_PROMPT = '''You are a savage, no-holds-barred dynasty fantasy football beat writer. \
Your job is to write power rankings blurbs that are smack-talky, opinionated, and funny. \
You pull no punches. You compare teams against each other, roast owners who are struggling, \
hype up the contenders, and call out delusion where you see it. You have strong opinions and \
you are not afraid to share them.

You will receive data for 10 dynasty fantasy football teams. Based on that data, you must:
1. Determine the correct power ranking order from 1 (best) to 10 (worst) based on dynasty \
value, outlook classification, tier composition, positional grades, and recent trades.
2. Assign each team a power_score from 0-100 reflecting your holistic, independent judgment \
of their current dynasty strength.
3. Write a 3-5 sentence blurb for each team that captures their current situation with \
personality and heat.
4. Return ONLY a valid JSON array with no markdown, no explanation, no preamble.

''' + DYNASTYZE_REFERENCE + '''

The JSON schema must be exactly:
[
  {
    "rank": 1,
    "team_name": "Team Name Here",
    "owner": "OwnerName",
    "outlook": "Contender",
    "power_score": 88,
    "blurb": "Your savage 3-5 sentence blurb here."
  }
]

Rules:
- rank 1 = best dynasty team, rank 10 = worst
- power_score is an integer from 0-100 reflecting your own holistic judgment of that team's \
current dynasty strength. It is NOT simply derived from rank — e.g. the #1 team does not \
automatically get 100. Score teams on their actual merits; ties, tight clusters, and large \
gaps between teams are all valid outcomes.
- Do NOT just follow the Value Rank order. Use your judgment based on ALL the data.
- Make cross-team comparisons and jokes where appropriate.
- Be specific: reference actual player names, pick counts, trade grades, positional weaknesses.
- Return ONLY the JSON array. Nothing else.'''


def format_team_block(t):
    top = ', '.join(
        f"{p['name']} ({p['position']}, {p['tier']}, KTC {p['ktc']:,})"
        for p in t['top_players']
    )
    tier_str = ', '.join(
        f"{tier}: {cnt}" for tier, cnt in t['tier_breakdown'].items()
    )
    depth_lines = []
    for pos in ['QB', 'RB', 'WR', 'TE']:
        players = t['positional_depth'].get(pos, [])
        if players:
            pnames = ', '.join(f"{p['name']} ({p['tier']})" for p in players[:3])
            depth_lines.append(f'  {pos}: {pnames}')
    depth_str = '\n'.join(depth_lines)

    trade_lines = []
    for tr in t['recent_trades'][:3]:
        if tr['surplus'] > 500:
            verdict = 'WIN'
        elif tr['surplus'] < -500:
            verdict = 'LOSS'
        else:
            verdict = 'FAIR'
        trade_lines.append(
            f'  [{tr["date"]} vs {tr["opponent"]} | {verdict} {tr["surplus"]:+,}] '
            f'Gave: {tr["gave"]} | Got: {tr["received"]}'
        )
    trades_str = '\n'.join(trade_lines) if trade_lines else '  No recent trades'

    g = t['positional_grades']
    prop = t['positional_proportion']

    return (
        f'--- {t["owner"]} (Team: {t["team_name"]}) ---\n'
        f'Outlook: {t["outlook"]}\n'
        f'Value Rank: #{t["value_rank"]}/10\n'
        f'Total Dynasty Value: {t["total_value"]:,} '
        f'(Players: {t["player_value"]:,}, Picks: {t["pick_value"]:,})\n'
        f'Cornerstones+Foundational: {t["cf_total"]}\n'
        f'Total 1st-round picks owned: {t["total_1sts"]}\n'
        f'Pos grades (avg starter KTC): QB {g["QB"]:,}, RB {g["RB"]:,}, '
        f'WR {g["WR"]:,}, TE {g["TE"]:,}\n'
        f'Value split: QB {prop["QB"]}%, RB {prop["RB"]}%, '
        f'WR {prop["WR"]}%, TE {prop["TE"]}%\n'
        f'\nTop 5 players by KTC:\n{top}\n'
        f'\nTier breakdown:\n{tier_str}\n'
        f'\nPositional depth (top 3 per pos):\n{depth_str}\n'
        f'\nRecent trades:\n{trades_str}\n'
    )


teams_text = '\n'.join(format_team_block(t) for t in teams_data)

user_message = (
    'Here is the data for all 10 teams in our dynasty league. Generate power rankings '
    'ranked 1 (best) to 10 (worst) with savage smack-talky blurbs. Determine your own '
    'ranking order based on ALL the data, not just Value Rank.\n\n'
    + teams_text
    + '\nReturn ONLY the JSON array. No markdown. No explanation. No preamble.'
)

print(f'Prompt length: {len(user_message):,} characters')
print('Calling Anthropic API...')

message = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=4096,
    system=SYSTEM_PROMPT,
    messages=[{'role': 'user', 'content': user_message}],
)

response_text = message.content[0].text
print(f'Response received ({len(response_text):,} chars) ✅')
print(f'Input tokens: {message.usage.input_tokens}, Output tokens: {message.usage.output_tokens}')

Prompt length: 16,512 characters
Calling Anthropic API...


Response received (8,205 chars) ✅
Input tokens: 8279, Output tokens: 2398


In [7]:
# ============================================================
# 7. Parse response and write output JSON
# ============================================================

# Strip accidental markdown code fences
cleaned = response_text.strip()
cleaned = re.sub(r'^```(?:json)?\n?', '', cleaned)
cleaned = re.sub(r'\n?```$', '', cleaned)
cleaned = cleaned.strip()

rankings = json.loads(cleaned)

# Clamp/coerce power_score defensively in case the model returns something malformed
for r in rankings:
    try:
        r['power_score'] = max(0, min(100, int(round(float(r.get('power_score', 0))))))
    except (TypeError, ValueError):
        r['power_score'] = 0

print(f'Parsed {len(rankings)} team rankings ✅')
for r in sorted(rankings, key=lambda x: x['rank']):
    print(f'  #{r["rank"]} {r["owner"]} ({r["outlook"]}) — score {r["power_score"]} — {r["blurb"][:80]}...')

output = {
    'generated_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'rankings': sorted(rankings, key=lambda x: x['rank']),
}

out_path = os.path.join(OUTPUT_DIR, 'power_rankings.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\n✅ Power rankings written to {out_path}')

Parsed 10 team rankings ✅
  #1 SvenMoney34 (Contender) — score 91 — Ja'Marr Chase, Jaxon Smith-Njigba, and Drake London at wide receiver? That's not...
  #2 ekleiner1123 (Contender) — score 87 — Drake Maye, Jahmyr Gibbs, Puka Nacua, and Trey McBride — four cornerstones just ...
  #3 jsinykin (Reload) — score 80 — Seven first-round picks and Caleb Williams at the helm — Siny is playing 4D ches...
  #4 gavinw20 (Reload) — score 76 — Josh Allen and Ashton Jeanty is arguably the best QB-RB combo in the entire leag...
  #5 GiantHawkTua (Contender) — score 70 — Amon-Ra St. Brown, Joe Burrow, Jalen Hurts, and Jonathan Taylor is a respectable...
  #6 nchernandez19 (Rebuild (future value)) — score 65 — Bijan Robinson and Brock Bowers are two of the most valuable dynasty assets in t...
  #7 Akracoon (Rebuild (future value)) — score 55 — Selling Ja'Marr Chase — one of the greatest dynasty WRs alive — for Ladd McConke...
  #8 GreyWaedekin27 (Window Contender) — score 50 — Justin Jefferson and CeeD